# Grant access (one-time, by hand)
- Gives each role group (and pipeline service principal) its access to one environment's catalog, per the access matrix.
- **How:** Environment → Mode = preview → Run all → check → Mode = apply → Run all. Safe to re-run. Not part of any job.
- Masking (column masks) is added on tables in Phase 5. Ownership transfer is a separate later step.

## 2. Repo constant and widgets

In [0]:
REPO_COMPANY = "bhg"          # fqf repo: "fqf"

dbutils.widgets.dropdown("env",  "dev",     ["dev", "test", "prod"], "Environment")
dbutils.widgets.dropdown("mode", "preview", ["preview", "apply"],    "Mode")

## 3. Access matrix

In [0]:
env     = dbutils.widgets.get("env")
mode    = dbutils.widgets.get("mode")
catalog = f"{REPO_COMPANY}_{env}"
SP      = f"sp_{REPO_COMPANY}_{env}"

R      = ["USE CATALOG", "USE SCHEMA", "SELECT", "READ VOLUME", "EXECUTE", "BROWSE"]
RW     = R + ["MODIFY", "CREATE TABLE", "CREATE VOLUME", "WRITE VOLUME", "CREATE FUNCTION", "CREATE MATERIALIZED VIEW"]
MANAGE = ["MANAGE"]
GOLD_R = ["USE SCHEMA", "SELECT", "EXECUTE"]      # applied on schema gold only

# (principal display name, scope, privileges)   scope = "catalog" or "gold"
MATRIX = {
    "dev":  [("lh_developer", "catalog", RW), ("lh_teamlead", "catalog", RW),
             ("lh_manager", "catalog", RW), ("lh_management", "catalog", R + MANAGE)],
    "test": [("lh_developer", "catalog", R), ("lh_teamlead", "catalog", R),
             ("lh_manager", "catalog", R + MANAGE), ("lh_bizmanager", "gold", GOLD_R),
             ("lh_management", "catalog", R + MANAGE), (SP, "catalog", RW)],
    "prod": [("lh_developer", "catalog", R), ("lh_teamlead", "catalog", R),
             ("lh_manager", "catalog", R), ("lh_bizmanager", "gold", GOLD_R),
             ("lh_management", "catalog", R + MANAGE), (SP, "catalog", RW)],
}
if REPO_COMPANY == "fqf":
    MATRIX["prod"].append(("sp_bhg_prod", "gold", GOLD_R))   # Gap 3: bhg prod job reads fqf_prod.gold for group gold
print(f"catalog={catalog} | mode={mode} | {len(MATRIX[env])} principals")

## 4. Resolve names (service principals are granted by their application ID)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

def resolve(name):
    if name.startswith("sp_"):
        sps = list(w.service_principals.list(filter=f"displayName eq '{name}'"))
        if not sps:
            raise Exception(f"Service principal '{name}' not found")
        return sps[0].application_id
    if not list(w.groups.list(filter=f"displayName eq '{name}'")):
        raise Exception(f"Group '{name}' not found")
    return name

IDS = {name: resolve(name) for name, _, _ in MATRIX[env]}
NAME_OF = {v: k for k, v in IDS.items()}
print("Resolved:", ", ".join(IDS))

## 5. Plan

In [0]:
stmts = []
for name, scope, privs in MATRIX[env]:
    p = IDS[name]
    if scope == "catalog":
        stmts.append((name, f"GRANT {', '.join(privs)} ON CATALOG {catalog} TO `{p}`"))
    else:  # gold only
        stmts.append((name, f"GRANT USE CATALOG, BROWSE ON CATALOG {catalog} TO `{p}`"))
        stmts.append((name, f"GRANT {', '.join(privs)} ON SCHEMA {catalog}.{scope} TO `{p}`"))

for name, s in stmts:
    print(f"[{name}] {s};")
print(f"\nPlan: {len(stmts)} GRANT statement(s)")

## 6. Apply 

In [0]:
if mode == "apply":
    for _, s in stmts:
        spark.sql(s)
    print(f"Applied {len(stmts)} GRANT statement(s) to {catalog}")
else:
    print("Preview only. Set Mode = apply and Run all.")

## 7. Verify (direct grants only), with Missing and Extra

In [0]:
norm = lambda a: a.replace("_", " ").upper()

def direct_grants(obj_type, obj):
    rows = spark.sql(f"SHOW GRANTS ON {obj_type} {obj}").collect()
    return {(NAME_OF.get(r.Principal, r.Principal), norm(r.ActionType))
            for r in rows if r.ObjectType.upper() == obj_type}

expected_cat, expected_gold = set(), set()
for name, scope, privs in MATRIX[env]:
    if scope == "catalog":
        expected_cat |= {(name, norm(x)) for x in privs}
    else:
        expected_cat  |= {(name, "USE CATALOG"), (name, "BROWSE")}
        expected_gold |= {(name, norm(x)) for x in privs}

actual_cat  = direct_grants("CATALOG", catalog)
actual_gold = direct_grants("SCHEMA", f"{catalog}.gold")

print("CATALOG  missing:", sorted(expected_cat - actual_cat) or "none")
print("CATALOG  extra  :", sorted(actual_cat - expected_cat) or "none")
print("GOLD     missing:", sorted(expected_gold - actual_gold) or "none")
print("GOLD     extra  :", sorted(actual_gold - expected_gold) or "none")